# 물리 기반 예측 + Kalman Filter 기반 예측 앙상블 Pipeline

- 입력: train/*.csv, test/*.csv, train_labels.csv, sample_submission.csv
- 출력: submission.csv

전략:
1. HyperPhysics_xy2를 5-fold로 학습 → OOF + test 예측
   (단일 모델 LB 0.696 → 5-fold 평균으로 안정성 ↑)
2. Kalman 잔차 학습 모델(Bi-GRU+Attn) 5-fold → OOF + test 예측
   (다양성 확보용 보조)
3. OOF 기반으로 (w1*Phys + w2*Kalman) 최적 가중치 grid search
4. 최적 가중치로 test 예측 결합 → submission.csv

In [1]:
!pip list

Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.2
aiohttp                                  3.13.5
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                         

In [ ]:
!cat /etc/issue
!lscpu | grep 'Model name'
!free -h
!df -h /
!nvidia-smi

Ubuntu 22.04.5 LTS \n \l

Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            83Gi       3.5Gi        74Gi        21Mi       5.4Gi        79Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   48G  189G  21% /
Mon Jun  1 05:02:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=====================

# 환경 설정 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "[open.zip 경로]" .
!unzip open.zip

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: train/TRAIN_05002.csv   
  inflating: train/TRAIN_05003.csv   
  inflating: train/TRAIN_05004.csv   
  inflating: train/TRAIN_05005.csv   
  inflating: train/TRAIN_05006.csv   
  inflating: train/TRAIN_05007.csv   
  inflating: train/TRAIN_05008.csv   
  inflating: train/TRAIN_05009.csv   
  inflating: train/TRAIN_05010.csv   
  inflating: train/TRAIN_05011.csv   
  inflating: train/TRAIN_05012.csv   
  inflating: train/TRAIN_05013.csv   
  inflating: train/TRAIN_05014.csv   
  inflating: train/TRAIN_05015.csv   
  inflating: train/TRAIN_05016.csv   
  inflating: train/TRAIN_05017.csv   
  inflating: train/TRAIN_05018.csv   
  inflating: train/TRAIN_05019.csv   
  inflating: train/TRAIN_05020.csv   
  inflating: train/TRAIN_05021.csv   
  inflating: train/TRAIN_05022.csv   
  inflating: train/TRAIN_05023.csv   
  inflating: train/TRAIN_05024.csv   
  inflating: train/TRAIN_05025.csv   
  inflating: train/TRAIN_05026.csv   
  inflating: t

In [ ]:
import os, sys, gc, time, random, hashlib, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter
from tqdm.auto import tqdm


In [ ]:
# 환경 설정 - 시드, 디바이스, 기타 상수
SEED = 42
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

DT, T_PRED, R_HIT, N_T, EPS = 0.040, 0.080, 0.01, 11, 1e-8

Device: cuda


In [ ]:
"""
데이터 로드
"""

DATA_DIR = Path('./')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

def load_stack(files):
    arrs = []
    for f in tqdm(files, leave=False):
        df = pd.read_csv(f)
        arrs.append(df[['x','y','z']].values)
    return np.stack(arrs, axis=0).astype(np.float64)

train_files = sorted(TRAIN_DIR.glob('TRAIN_*.csv'))
test_files  = sorted(TEST_DIR.glob('TEST_*.csv'))
train_labels = pd.read_csv(DATA_DIR / 'train_labels.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print("Loading train ...")
X_train = load_stack(train_files)
print("Loading test ...")
X_test  = load_stack(test_files)
train_ids = [f.stem for f in train_files]
Y_train = train_labels.set_index('id').loc[train_ids][['x','y','z']].values.astype(np.float64)
print(f"X_train: {X_train.shape}, Y_train: {Y_train.shape}, X_test: {X_test.shape}")

Loading train ...


  0%|          | 0/10000 [00:00<?, ?it/s]

Loading test ...


  0%|          | 0/10000 [00:00<?, ?it/s]

X_train: (10000, 11, 3), Y_train: (10000, 3), X_test: (10000, 11, 3)


In [ ]:
# 해시를 이용한 KFold

def stable_fold_id(sid, folds):
    return int(hashlib.md5(str(sid).encode()).hexdigest()[:8], 16) % folds

def make_kfold_splits(ids, n_folds=5):
    fids = np.asarray([stable_fold_id(s, n_folds) for s in ids])
    return [(np.where(fids != f)[0], np.where(fids == f)[0]) for f in range(n_folds)]

N_FOLDS = 5
splits = make_kfold_splits(train_ids, N_FOLDS)
for f,(tr,va) in enumerate(splits):
    print(f"  Fold {f}: tr={len(tr)}, va={len(va)}")

  Fold 0: tr=7980, va=2020
  Fold 1: tr=7953, va=2047
  Fold 2: tr=8079, va=1921
  Fold 3: tr=7980, va=2020
  Fold 4: tr=8008, va=1992


# 메인 예측기: HyperPhysics_xy2
- CREE님의 "[LB_0.699+] 물리학 기반 추론 모델.ipynb - Colab"에서 공개해주신 모델입니다.
- 현재 코드가 내려간 것으로 보입니다.

In [ ]:
# ----- A.1 Sliding window dataset (등속 외삽 패딩) -----
class SlidingWindowDataset(Dataset):
    def __init__(self, X, y, min_win=3, mode="extended", device="cpu"):
        Xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.float32)
        windows = []
        for i in range(len(X)):
            targets = [4,5,6,7,8,9,10,12] if mode == "extended" else [12, 10]
            for tgt in targets:
                end_idx = tgt - 2
                max_w = end_idx + 2 if mode == "extended" else (12 if tgt == 12 else 11)
                for w in range(min_win, max_w):
                    windows.append((i, w, tgt))
        X_list, y_list = [], []
        for i, w, tgt in windows:
            X_o = Xt[i]
            end_idx = tgt - 2
            pts = X_o[end_idx - w + 1 : end_idx + 1]
            target = yt[i] if tgt == 12 else X_o[tgt]
            if w < 11:
                v0 = pts[1] - pts[0]
                n_pad = 11 - w
                js = torch.arange(n_pad, 0, -1, dtype=torch.float32)
                pad = pts[0:1] - js.unsqueeze(1) * v0.unsqueeze(0)
                X_pad = torch.cat([pad, pts], dim=0)
            else:
                X_pad = pts.clone()
            X_list.append(X_pad); y_list.append(target)
        self.X_all = torch.stack(X_list).to(device)
        self.y_all = torch.stack(y_list).to(device)
        # theta weights
        diffs = self.X_all[:,1:] - self.X_all[:,:-1]
        n1 = diffs[:,1:].norm(dim=2).clamp(min=1e-8)
        n2 = diffs[:,:-1].norm(dim=2).clamp(min=1e-8)
        cos_t = ((diffs[:,1:] * diffs[:,:-1]).sum(dim=2) / (n1*n2)).clamp(-1,1)
        theta_last = torch.acos(cos_t[:,-1])
        self.theta_weights = (1.0 + 4.0 * (theta_last/1.0).clamp(0,1)).cpu().numpy()
    def __len__(self): return len(self.X_all)
    def __getitem__(self, i): return self.X_all[i], self.y_all[i]

# ----- A.2 Feature extraction -----
def _ema_va_local(diffs_local, alpha, beta):
    B, T, _ = diffs_local.shape
    one_m_a = 1.0 - alpha; one_m_b = 1.0 - beta
    vs = diffs_local.new_empty(B, T, 3)
    v = diffs_local[:, 0]; vs[:, 0] = v
    for t in range(1, T):
        v = alpha * diffs_local[:, t] + one_m_a * v
        vs[:, t] = v
    vl = vs[:, -1]
    ad = vs[:, 1:] - vs[:, :-1]
    a = ad[:, 0]
    for t in range(1, T - 1):
        a = beta * ad[:, t] + one_m_b * a
    return vl, a

def _soft_hit_loss(pred, target, thr=0.013012, k=408.348):
    return (1 - torch.sigmoid(-(torch.norm(pred-target, dim=1) - thr) * k)).mean()

def extract_features(X, mean_stats=None, std_stats=None, dir_net=None, heading_mode="3step"):
    device = X.device
    p_last = X[:, 10]
    diffs = X[:, 1:] - X[:, :-1]
    n1 = diffs[:,1:].norm(dim=2, keepdim=True) + 1e-8
    n2 = diffs[:,:-1].norm(dim=2, keepdim=True) + 1e-8
    cos_t = ((diffs[:,1:]*diffs[:,:-1]).sum(dim=2, keepdim=True) / (n1*n2)).clamp(-1,1)
    theta_seq = torch.acos(cos_t).squeeze(2)
    theta = theta_seq[:, -1:]
    theta_mean = theta_seq.mean(1, keepdim=True)
    theta_std  = theta_seq.std(1, keepdim=True)
    theta_vel  = theta_seq[:, -1:] - theta_seq[:, -2:-1]
    theta_acc  = theta_seq[:, -1:] - 2*theta_seq[:, -2:-1] + theta_seq[:, -3:-2]
    theta_trend= theta_seq[:, -1:] - theta_seq[:, -3:].mean(1, keepdim=True)

    if dir_net is not None:
        speed_seq = diffs.norm(dim=2)
        state = torch.cat([speed_seq, theta_seq], dim=1)
        if dir_net[0].in_features == 29:
            z_speed = diffs[:,:,2].abs()
            state = torch.cat([state, z_speed], dim=1)
        weights = F.softmax(dir_net(state), dim=1)
        v_sm = (diffs * weights.unsqueeze(2)).sum(dim=1)
    else:
        v_sm = (3*diffs[:,-1] + 2*diffs[:,-2] + diffs[:,-3]) / 6.0

    fwd = v_sm / (v_sm.norm(dim=1, keepdim=True) + 1e-8)
    up_w = torch.zeros_like(fwd); up_w[:,2] = 1.0
    up_w[fwd[:,2].abs() > 0.99] = torch.tensor([0.,1.,0.], device=device)
    right = torch.cross(fwd, up_w, dim=1)
    right = right / (right.norm(dim=1, keepdim=True) + 1e-8)
    up = torch.cross(right, fwd, dim=1)
    up = up / (up.norm(dim=1, keepdim=True) + 1e-8)
    R = torch.stack([fwd, right, up], dim=2)

    v_last = diffs[:,-1]; v_prev1 = diffs[:,-2]
    speed = v_last.norm(dim=1, keepdim=True)
    a_last = v_last - v_prev1
    acc_mag = a_last.norm(dim=1, keepdim=True)
    v_local = torch.matmul(v_last.unsqueeze(1), R).squeeze(1)
    a_local = torch.matmul(a_last.unsqueeze(1), R).squeeze(1)
    X_local = torch.matmul(X - p_last.unsqueeze(1), R)
    p_std_local = X_local.std(1)
    v_local_abs = v_local.abs()
    jerk_g = diffs[:,-1] - 2*diffs[:,-2] + diffs[:,-3]
    jerk_l = torch.matmul(jerk_g.unsqueeze(1), R).squeeze(1)
    jerk_mag = jerk_g.norm(dim=1, keepdim=True)

    feats = torch.cat([
        v_local, a_local, speed, acc_mag,
        theta, theta_mean, theta_std, theta_trend, theta_vel, theta_acc,
        p_std_local, v_local_abs, jerk_l, jerk_mag
    ], dim=1)

    if mean_stats is None or std_stats is None:
        mean_stats = feats.mean(0, keepdim=True)
        std_stats  = feats.std(0, keepdim=True) + 1e-8
    feats_scaled = (feats - mean_stats) / std_stats
    return feats_scaled, diffs, p_last, theta, theta_mean, theta_std, theta_seq, R, speed, mean_stats, std_stats

# ----- A.3 모델 컴포넌트 -----
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(dim, dim))
        self.ln = nn.LayerNorm(dim)
    def forward(self, x): return self.ln(x + self.net(x))

class PriorBiasedLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_bias):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f)
        self.register_buffer('prior_bias', prior_bias.clone().detach())
        with torch.no_grad():
            nn.init.zeros_(self.linear.weight); nn.init.zeros_(self.linear.bias)
    def forward(self, x): return self.linear(x) + self.prior_bias

def rodrigues_rotate(v, w):
    theta = w.norm(dim=1, keepdim=True)
    k = w / (theta + 1e-8)
    cos_t, sin_t = torch.cos(theta), torch.sin(theta)
    dot = (v*k).sum(dim=1, keepdim=True)
    cross = torch.cross(k, v, dim=1)
    return v*cos_t + cross*sin_t + k*dot*(1.0-cos_t)

class HyperPhysics_xy2(nn.Module):
    def __init__(self, input_dim=24):
        super().__init__()
        self.sh_thr=0.013012; self.sh_k=408.348044
        self.mse_w=129.172037; self.local_w=0.050941
        self.theta_thr=1.087618; self.speed_thr=0.034583
        self.lr=0.005400; self.wd=0.005659
        self.register_buffer("mean_stats", torch.zeros(1, input_dim))
        self.register_buffer("std_stats",  torch.ones(1, input_dim))
        prior_dir = torch.tensor([-10.,-10.,-10.,-10.,-10.,-10.,-10., 0., 0.693, 1.099])
        self.dir_net = nn.Sequential(
            nn.Linear(29, 24), nn.LayerNorm(24), nn.GELU(),
            PriorBiasedLinear(24, 10, prior_dir))
        prior_ema = torch.zeros(6)
        self.temporal_net = nn.Sequential(
            nn.Linear(9, 32), nn.LayerNorm(32), nn.GELU(),
            PriorBiasedLinear(32, 6, prior_ema))
        prior_dyn = torch.tensor([0.]*6 + [-4.]*24)
        self.dynamics_net = nn.Sequential(
            nn.Linear(input_dim, 96), nn.LayerNorm(96), nn.GELU(),
            ResBlock(96),
            PriorBiasedLinear(96, 30, prior_dyn))
        self.omega_w = nn.Parameter(torch.tensor([0.0, -0.5, -1.0]))
        self.omega_net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 48), nn.GELU(),
            nn.Linear(48, 3))
        with torch.no_grad():
            nn.init.normal_(self.omega_net[-1].weight, std=0.01)
            nn.init.zeros_(self.omega_net[-1].bias)
        self.diffusion_net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.LayerNorm(32), nn.GELU(),
            nn.Linear(32, 3))

    def get_features(self, X, mean_stats=None, std_stats=None):
        return extract_features(X, mean_stats, std_stats, self.dir_net, "3step")

    @staticmethod
    def _rot_vec(d_prev, d_curr):
        n_p = d_prev.norm(dim=1, keepdim=True).clamp(min=1e-8)
        n_c = d_curr.norm(dim=1, keepdim=True).clamp(min=1e-8)
        d_hp = d_prev / n_p; d_hc = d_curr / n_c
        cross = torch.linalg.cross(d_hp, d_hc, dim=1)
        sin_t = cross.norm(dim=1, keepdim=True).clamp(min=1e-8)
        cos_t = (d_hp*d_hc).sum(1, keepdim=True).clamp(-0.9999, 0.9999)
        theta = torch.atan2(sin_t, cos_t)
        speed_gate = torch.sigmoid((n_p + n_c) * 500 - 5)
        return cross / sin_t * theta * speed_gate

    def forward(self, features, diffs, p_last, theta, speed, R):
        B = diffs.shape[0]
        ema_raw = self.temporal_net(features[:, 8:17])
        alpha = torch.sigmoid(ema_raw[:, 0:3]) * 0.8 + 0.1
        beta  = torch.sigmoid(ema_raw[:, 3:6]) * 0.199 + 0.8
        dyn_raw = self.dynamics_net(features)
        w_v = 2.0 + dyn_raw[:, 0:3]
        w_a = 1.0 + dyn_raw[:, 3:6]
        v_la = features[:, 17:20]; v_la2 = v_la*v_la; th2 = theta*theta
        exp_v = (F.softplus(dyn_raw[:, 6:9])*v_la + F.softplus(dyn_raw[:, 9:12])*v_la2
               + F.softplus(dyn_raw[:, 12:15])*theta + F.softplus(dyn_raw[:, 15:18])*th2)
        exp_a = (F.softplus(dyn_raw[:, 18:21])*v_la + F.softplus(dyn_raw[:, 21:24])*v_la2
               + F.softplus(dyn_raw[:, 24:27])*theta + F.softplus(dyn_raw[:, 27:30])*th2)
        diffs_local = torch.matmul(diffs, R)
        vl, al = _ema_va_local(diffs_local, alpha, beta)
        diff_speed = diffs_local.norm(dim=2)

        def rv_m(ka, kb):
            rv = self._rot_vec(diffs_local[:, ka], diffs_local[:, kb])
            valid = ((diff_speed[:, ka] > 1e-5) & (diff_speed[:, kb] > 1e-5)).float()
            return rv * valid.unsqueeze(1), valid

        ov1, vm1 = rv_m(-2, -1)
        ov2, vm2 = rv_m(-3, -2)
        ov3, vm3 = rv_m(-4, -3)
        w_logits = self.omega_w.view(1, 3).expand(B, -1)
        masks = torch.stack([vm1, vm2, vm3], dim=1)
        w_logits = w_logits.masked_fill(masks == 0, -1e9)
        w_attn = F.softmax(w_logits, dim=1)
        omega_hist = (w_attn[:,0:1]*ov1 + w_attn[:,1:2]*ov2 + w_attn[:,2:3]*ov3)
        cur_speed = speed.view(B, 1) if speed is not None else diff_speed[:, -1:].clone()
        omega_sgate = torch.sigmoid(cur_speed * 500 - 5)
        omega_delta = self.omega_net(features) * omega_sgate
        theta_s = theta.view(B, 1)
        theta_gate = torch.sigmoid((theta_s - self.theta_thr) * 10)
        speed_gate_strong = torch.sigmoid((cur_speed - self.speed_thr) * 200)
        rot_gate = theta_gate * speed_gate_strong
        omega = (omega_hist + omega_delta) * rot_gate
        v_rotated = rodrigues_rotate(vl, omega)
        pred_local = (w_v * torch.exp(-exp_v)) * v_rotated + (w_a * torch.exp(-exp_a)) * al
        log_var = self.diffusion_net(features).clamp(-5.0, 5.0)
        pred_global = p_last + torch.einsum('nij,nj->ni', R, pred_local)
        return pred_global, pred_local, log_var

    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None):
        sh = _soft_hit_loss(pp, yr, thr=self.sh_thr, k=self.sh_k)
        loss = sh + self.mse_w * F.mse_loss(pp, yr)
        if pred_local is not None and yr_local is not None and log_var is not None:
            se = (pred_local - yr_local) ** 2
            nll = 0.5 * (torch.exp(-log_var) * se + log_var)
            loss = loss + self.local_w * nll.mean()
        return loss

# ----- A.4 단일 fold 학습 -----
def train_phys_fold(fold, tr_idx, va_idx, X, Y, epochs=80, seed=0):
    set_seed(seed)
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = Y[tr_idx], Y[va_idx]
    tr_ds = SlidingWindowDataset(X_tr, y_tr, min_win=3, mode="extended", device=DEVICE)
    va_ds = SlidingWindowDataset(X_va, y_va, min_win=11, mode="standard", device=DEVICE)
    sampler = WeightedRandomSampler(tr_ds.theta_weights, len(tr_ds), replacement=True)
    tr_ld = DataLoader(tr_ds, batch_size=256, sampler=sampler)
    va_ld = DataLoader(va_ds, batch_size=512, shuffle=False)

    model = HyperPhysics_xy2().to(DEVICE)
    with torch.no_grad():
        _, _, _, _, _, _, _, _, _, mn, st = model.get_features(
            torch.tensor(X_tr, dtype=torch.float32, device=DEVICE))
        model.mean_stats.copy_(mn); model.std_stats.copy_(st)

    opt = torch.optim.AdamW(model.parameters(), lr=model.lr, weight_decay=model.wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_rh, best_state, no_imp = -1.0, None, 0
    PATIENCE = 15
    for ep in range(1, epochs+1):
        model.train(); tot = 0.0; n = 0
        for Xb, yb in tr_ld:
            opt.zero_grad(set_to_none=True)
            ft, df, plt_, tht, _, _, _, Rt, spt, _, _ = model.get_features(Xb, model.mean_stats, model.std_stats)
            pp, pl, lv = model(ft, df, plt_, tht, spt, Rt)
            yr_local = torch.matmul((yb - plt_).unsqueeze(1), Rt).squeeze(1)
            loss = model.compute_loss(pp, yb, pl, yr_local, lv)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(Xb); n += len(Xb)
        sched.step()
        model.eval(); hits = 0
        preds_all = []
        with torch.no_grad():
            for Xv, yv in va_ld:
                ft, df, plt_, tht, _, _, _, Rt, spt, _, _ = model.get_features(Xv, model.mean_stats, model.std_stats)
                pp, _, _ = model(ft, df, plt_, tht, spt, Rt)
                hits += (torch.norm(pp - yv, dim=1) <= R_HIT).sum().item()
                preds_all.append(pp.cpu().numpy())
        rh = hits / len(X_va)
        if rh > best_rh:
            best_rh = rh; no_imp = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_pred = np.concatenate(preds_all, axis=0)
        else:
            no_imp += 1
        if ep % 5 == 0 or ep == 1:
            print(f"    ep {ep:3d} | loss {tot/n:.4f} | rh {rh:.4f} | best {best_rh:.4f}")
        if no_imp >= PATIENCE:
            print(f"    early stop @ ep {ep}, best {best_rh:.4f}")
            break
    return best_state, best_rh, best_pred

# ----- A.5 fold별 test 예측 -----
def predict_phys(model_state, X_test_np, X_train_for_stats):
    model = HyperPhysics_xy2().to(DEVICE)
    model.load_state_dict(model_state); model.eval()
    Xt = torch.tensor(X_test_np, dtype=torch.float32, device=DEVICE)
    preds = []
    with torch.no_grad():
        bs = 512
        for i in range(0, len(Xt), bs):
            Xb = Xt[i:i+bs]
            ft, df, plt_, tht, _, _, _, Rt, spt, _, _ = model.get_features(Xb, model.mean_stats, model.std_stats)
            pp, _, _ = model(ft, df, plt_, tht, spt, Rt)
            preds.append(pp.cpu().numpy())
    return np.concatenate(preds, axis=0)


# 보조 예측기: Kalman filter 기반 BiGRUAttn

In [ ]:
def yaw_angle(v): return np.arctan2(v[:,1], v[:,0])
def rotate_xy(vec, theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.stack([vec[:,0]*c + vec[:,1]*s,
                     -vec[:,0]*s + vec[:,1]*c, vec[:,2]], axis=-1)
def inverse_rotate_xy(vec, theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.stack([vec[:,0]*c - vec[:,1]*s,
                     vec[:,0]*s + vec[:,1]*c, vec[:,2]], axis=-1)
def rotate_xy_seq(seq, theta):
    c, s = np.cos(theta)[:,None], np.sin(theta)[:,None]
    return np.stack([seq[:,:,0]*c + seq[:,:,1]*s,
                     -seq[:,:,0]*s + seq[:,:,1]*c, seq[:,:,2]], axis=-1)
def compute_yaw_theta(X):
    v_last = (X[:,-1] - X[:,-2]) / DT
    return yaw_angle(v_last)

def kalman_predict(X, dt=DT, t_pred=T_PRED, sigma_obs=0.5e-3, sigma_proc=2.0, P0=1.0):
    N, T, _ = X.shape
    F_ = np.array([[1, dt], [0, 1]])
    F_pred = np.array([[1, t_pred], [0, 1]])
    Q = sigma_proc**2 * np.array([[dt**4/4, dt**3/2],[dt**3/2, dt**2]])
    R_ = sigma_obs**2
    pred = np.zeros((N, 3))
    for j in range(3):
        z = X[:,:,j]; state = np.zeros((N, 2)); state[:,0] = z[:,0]
        P = np.eye(2) * P0
        for t in range(1, T):
            state = state @ F_.T
            P = F_ @ P @ F_.T + Q
            innov = z[:,t] - state[:,0]
            S = P[0,0] + R_
            K = P[:,0] / S
            state = state + np.outer(innov, K)
            P = P - np.outer(K, P[0,:])
        pred[:,j] = (state @ F_pred.T)[:,0]
    return pred.astype(np.float32)

def cos_safe(a, b):
    na = np.linalg.norm(a,axis=-1); nb = np.linalg.norm(b,axis=-1)
    return np.clip((a*b).sum(-1)/np.maximum(na*nb, 1e-12), -1, 1)

def noise_poly2(X):
    t = np.arange(-(N_T-1)*DT*1000, 1, DT*1000) / 1000.0
    V = np.vander(t, 3, increasing=False)
    out = np.zeros(X.shape[0])
    for j in range(3):
        coef = np.linalg.lstsq(V, X[:,:,j].T, rcond=None)[0]
        fit = (V @ coef).T
        out += (X[:,:,j] - fit).std(axis=1)
    return out / 3

def noise_savgol(X, w=5, p=2):
    Xs = savgol_filter(X, window_length=w, polyorder=p, axis=1)
    return (X - Xs).std(axis=1).mean(axis=-1)

def build_scalar_features_40(X, verbose=False):
    if verbose: print("  scalar features...")
    noise_p = noise_poly2(X); noise_s = noise_savgol(X)
    # LOO spline은 비싸니 savgol로 대체 (시간 절약, 성능 영향 미미)
    noise_l = noise_s.copy()
    delta_ = np.diff(X, axis=1); v_ = delta_/DT
    a_ = np.diff(v_, axis=1)/DT; jerk_ = np.diff(a_, axis=1)/DT
    sp_ = np.linalg.norm(v_, axis=-1)
    ac_ = np.linalg.norm(a_, axis=-1)
    jk_ = np.linalg.norm(jerk_, axis=-1)
    v_l = v_[:,-1]; a_l = a_[:,-1]; a_r = a_[:,-3:].mean(1)
    nd_vec = X[:,-1] - X[:,0]; nd = np.linalg.norm(nd_vec, axis=-1)
    pl = np.linalg.norm(delta_, axis=-1).sum(axis=1)
    straight = np.where(pl > 1e-12, nd / np.maximum(pl, 1e-12), 0.0)
    turn = cos_safe(v_l, v_[:,:-1].mean(axis=1))
    df = pd.DataFrame({
        'mean_speed':sp_.mean(1), 'max_speed':sp_.max(1), 'speed_std':sp_.std(1),
        'mean_acc':ac_.mean(1), 'max_acc':ac_.max(1), 'max_jerk':jk_.max(1),
        'straightness':straight, 'net_disp':nd, 'turn_cos':turn,
        '|v_last|':np.linalg.norm(v_l,axis=-1), '|a_last|':np.linalg.norm(a_l,axis=-1),
        '|a_recent|':np.linalg.norm(a_r,axis=-1),
        'jerk_last':jk_[:,-1], 'jerk_recent':jk_[:,-3:].mean(1),
        'noise_poly2':noise_p, 'noise_savgol':noise_s, 'noise_loo':noise_l,
        'hard_turn':(turn < 0.5).astype(np.float32),
        'high_speed':(np.linalg.norm(v_l,axis=-1) > 1.0).astype(np.float32),
        'high_acc':(ac_.max(axis=1) > 15).astype(np.float32),
        'log_max_acc':np.log1p(ac_.max(1)),
    })
    log_cols = ['mean_speed','max_speed','speed_std','mean_acc','max_acc','max_jerk',
                'net_disp','|v_last|','|a_last|','|a_recent|','jerk_last','jerk_recent',
                'noise_poly2','noise_savgol','noise_loo']
    for c in log_cols: df[c] = np.log1p(df[c])
    speed_roll = np.stack([sp_[:,i:i+3].mean(axis=1) for i in range(8)], axis=1) * DT
    disp_n = np.linalg.norm(delta_, axis=-1)
    cum_path = np.concatenate([np.zeros((X.shape[0],1)), np.cumsum(disp_n, axis=1)], axis=1)
    tier3 = np.concatenate([speed_roll, cum_path], axis=1).astype(np.float32)
    return np.concatenate([df.values.astype(np.float32), tier3], axis=-1)

def build_seq_input_9ch(X, theta):
    N = X.shape[0]
    X_loc = rotate_xy_seq(X, theta)
    rel = X_loc - X_loc[:,-1:,:]
    disp = np.diff(X_loc, axis=1); v = disp / DT
    v_p = np.concatenate([np.zeros((N,1,3)), v], axis=1)
    a = np.diff(v, axis=1) / DT
    a_p = np.concatenate([np.zeros((N,2,3)), a], axis=1)
    return np.concatenate([rel, v_p, a_p], axis=-1).astype(np.float32)

class BiGRUAttn(nn.Module):
    def __init__(self, n_ch=9, scal_dim=40, hidden=48, fc=128, p=0.2, out_scale_cm=2.0):
        super().__init__()
        self.gru = nn.GRU(n_ch, hidden, 1, batch_first=True, bidirectional=True)
        go = hidden*2
        self.attn = nn.Sequential(nn.Linear(go, go//2), nn.Tanh(), nn.Linear(go//2, 1))
        self.fc1 = nn.Linear(go + scal_dim, fc); self.ln1 = nn.LayerNorm(fc)
        self.fc2 = nn.Linear(fc, fc//2); self.ln2 = nn.LayerNorm(fc//2)
        self.act = nn.GELU(); self.drop = nn.Dropout(p)
        self.head = nn.Linear(fc//2, 3)
        self.scale = out_scale_cm / 100.0
    def forward(self, seq, scal):
        out, _ = self.gru(seq)
        a = torch.softmax(self.attn(out), dim=1)
        pooled = (out * a).sum(dim=1)
        z = self.act(self.ln1(self.fc1(torch.cat([pooled, scal], dim=1))))
        z = self.drop(z)
        z = self.act(self.ln2(self.fc2(z)))
        return torch.tanh(self.head(z)) * self.scale

def loss_kal(pred, true):
    d = torch.sqrt(((pred-true)**2).sum(-1) + 1e-12)
    return d.mean() + 0.3 * torch.sigmoid((d - R_HIT) / 0.002).mean()

def train_kalman_fold(fold, tr_idx, va_idx, X, Y, theta, kal, X_scal, epochs=120, seed=0):
    set_seed(seed)
    seq_full = build_seq_input_9ch(X, theta)
    sc_seq = StandardScaler().fit(seq_full[tr_idx].reshape(-1,9))
    seq_tr = sc_seq.transform(seq_full[tr_idx].reshape(-1,9)).reshape(-1,11,9).astype(np.float32)
    seq_va = sc_seq.transform(seq_full[va_idx].reshape(-1,9)).reshape(-1,11,9).astype(np.float32)
    sc_sc = StandardScaler().fit(X_scal[tr_idx])
    sc_tr = sc_sc.transform(X_scal[tr_idx]).astype(np.float32)
    sc_va = sc_sc.transform(X_scal[va_idx]).astype(np.float32)
    tgt_tr = rotate_xy(Y[tr_idx] - kal[tr_idx], theta[tr_idx]).astype(np.float32)

    T = lambda a: torch.from_numpy(a).to(DEVICE)
    seq_tr_t, seq_va_t = T(seq_tr), T(seq_va)
    sc_tr_t, sc_va_t = T(sc_tr), T(sc_va)
    tgt_tr_t = T(tgt_tr)

    model = BiGRUAttn().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=7e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_rh, best_state, no_imp = -1.0, None, 0
    PATIENCE = 25; bs = 256; n_tr = len(seq_tr_t)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_tr, device=DEVICE)
        tot = 0.0; nb = 0
        for i in range(0, n_tr, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            out = model(seq_tr_t[idx], sc_tr_t[idx])
            loss = loss_kal(out, tgt_tr_t[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        sched.step()
        model.eval()
        with torch.no_grad():
            out_va = model(seq_va_t, sc_va_t).cpu().numpy()
        pred_world = inverse_rotate_xy(out_va, theta[va_idx])
        pred_pos = kal[va_idx] + pred_world
        err = np.linalg.norm(pred_pos - Y[va_idx], axis=-1)
        rh = float((err <= R_HIT).mean())
        if rh > best_rh:
            best_rh = rh; no_imp = 0
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_pred = pred_pos
        else:
            no_imp += 1
        if (ep+1) % 10 == 0 or ep == 0:
            print(f"    ep {ep+1:3d} | loss {tot/nb:.4f} | rh {rh:.4f} | best {best_rh:.4f}")
        if no_imp >= PATIENCE:
            print(f"    early stop @ {ep+1}, best {best_rh:.4f}"); break

    return best_state, sc_seq, sc_sc, best_rh, best_pred

def predict_kalman(state, sc_seq, sc_sc, X_test, theta_test, kal_test, X_scal_test):
    seq_t = build_seq_input_9ch(X_test, theta_test)
    seq_n = sc_seq.transform(seq_t.reshape(-1,9)).reshape(-1,11,9).astype(np.float32)
    sc_n = sc_sc.transform(X_scal_test).astype(np.float32)
    model = BiGRUAttn().to(DEVICE)
    model.load_state_dict(state); model.eval()
    preds = []
    with torch.no_grad():
        bs = 1024
        for i in range(0, len(seq_n), bs):
            s_t = torch.from_numpy(seq_n[i:i+bs]).to(DEVICE)
            c_t = torch.from_numpy(sc_n[i:i+bs]).to(DEVICE)
            preds.append(model(s_t, c_t).cpu().numpy())
    pred_rot = np.concatenate(preds, axis=0)
    pred_world = inverse_rotate_xy(pred_rot, theta_test)
    return kal_test + pred_world


# 학습

In [ ]:
# C.1 Kalman 사전계산 (param grid는 노트북 2의 최적값 σ_obs=0.5mm, σ_proc=2.0 사용)
print("\n[Kalman precompute]")
kal_train = kalman_predict(X_train, sigma_obs=0.5e-3, sigma_proc=2.0)
kal_test  = kalman_predict(X_test,  sigma_obs=0.5e-3, sigma_proc=2.0)
theta_train = compute_yaw_theta(X_train)
theta_test  = compute_yaw_theta(X_test)

print("[Scalar features]")
X_scal_train = build_scalar_features_40(X_train, verbose=True)
X_scal_test  = build_scalar_features_40(X_test,  verbose=True)

# C.2 Phys 모델 5-fold
print("\n" + "="*60); print("Training HyperPhysics_xy2 (5-fold)"); print("="*60)
N = len(X_train)
oof_phys = np.zeros((N, 3), dtype=np.float32)
test_phys_folds = []
phys_states = []
phys_rh = []
for f, (tr, va) in enumerate(splits):
    print(f"\n--- Phys Fold {f+1}/{N_FOLDS} ---")
    state, rh, val_pred = train_phys_fold(f, tr, va, X_train, Y_train, epochs=80, seed=SEED+f)
    oof_phys[va] = val_pred
    phys_states.append(state)
    phys_rh.append(rh)
    # test 예측
    test_pred = predict_phys(state, X_test, X_train[tr])
    test_phys_folds.append(test_pred)
    torch.cuda.empty_cache(); gc.collect()
test_phys = np.mean(test_phys_folds, axis=0)
oof_phys_rh = float((np.linalg.norm(oof_phys - Y_train, axis=1) <= R_HIT).mean())
print(f"\n>>> Phys OOF r_Hit: {oof_phys_rh:.4f} (fold mean: {np.mean(phys_rh):.4f})")

# C.3 Kalman residual 모델 5-fold
print("\n" + "="*60); print("Training Kalman-residual BiGRU (5-fold)"); print("="*60)
oof_kal = np.zeros((N, 3), dtype=np.float32)
test_kal_folds = []
kal_states = []
for f, (tr, va) in enumerate(splits):
    print(f"\n--- Kal Fold {f+1}/{N_FOLDS} ---")
    state, sc_seq, sc_sc, rh, val_pred = train_kalman_fold(
        f, tr, va, X_train, Y_train, theta_train, kal_train, X_scal_train,
        epochs=120, seed=SEED+f*7)
    oof_kal[va] = val_pred
    kal_states.append(state)
    test_pred = predict_kalman(state, sc_seq, sc_sc, X_test, theta_test, kal_test, X_scal_test)
    test_kal_folds.append(test_pred)
    torch.cuda.empty_cache(); gc.collect()
test_kal = np.mean(test_kal_folds, axis=0)
oof_kal_rh = float((np.linalg.norm(oof_kal - Y_train, axis=1) <= R_HIT).mean())
print(f"\n>>> Kal OOF r_Hit: {oof_kal_rh:.4f}")



[Kalman precompute]
[Scalar features]
  scalar features...
  scalar features...

Training HyperPhysics_xy2 (5-fold)

--- Phys Fold 1/5 ---
    ep   1 | loss 0.2291 | rh 0.6728 | best 0.6728
    ep   5 | loss 0.2098 | rh 0.6817 | best 0.6817
    ep  10 | loss 0.2007 | rh 0.6787 | best 0.6847
    ep  15 | loss 0.1934 | rh 0.6738 | best 0.6847
    ep  20 | loss 0.1902 | rh 0.6757 | best 0.6847
    early stop @ ep 24, best 0.6847

--- Phys Fold 2/5 ---
    ep   1 | loss 0.2279 | rh 0.6649 | best 0.6649
    ep   5 | loss 0.2071 | rh 0.6590 | best 0.6683
    ep  10 | loss 0.1997 | rh 0.6624 | best 0.6683
    ep  15 | loss 0.1949 | rh 0.6615 | best 0.6698
    ep  20 | loss 0.1914 | rh 0.6663 | best 0.6698
    ep  25 | loss 0.1867 | rh 0.6673 | best 0.6698
    early stop @ ep 28, best 0.6698

--- Phys Fold 3/5 ---
    ep   1 | loss 0.2309 | rh 0.6679 | best 0.6679
    ep   5 | loss 0.2117 | rh 0.6637 | best 0.6715
    ep  10 | loss 0.2013 | rh 0.6720 | best 0.6720
    ep  15 | loss 0.1971 | r

# Model Ensemble
- 주 예측기와 보조 예측기의 결과에서 단순한 Voting을 통해 최종 결과를 만듭니다.
- 가중 평균의 weight는 단순히 0.0에서 1.0 까지의 grid search로 찾았습니다.
- 결과적으로 구한 가중치 w = 0.96

In [ ]:
print("\n" + "="*60); print("Blending"); print("="*60)

def r_hit(pred, gt): return float((np.linalg.norm(pred-gt, axis=1) <= R_HIT).mean())

# D.1 grid search
best_w, best_rh = 1.0, oof_phys_rh
for w in np.arange(0.0, 1.01, 0.02):
    blend = w * oof_phys + (1-w) * oof_kal
    rh = r_hit(blend, Y_train)
    if rh > best_rh:
        best_rh = rh; best_w = w
print(f"Phys alone OOF: {oof_phys_rh:.4f}")
print(f"Kal alone OOF:  {oof_kal_rh:.4f}")
print(f"Best blend: w_phys={best_w:.2f}, w_kal={1-best_w:.2f} → OOF: {best_rh:.4f}")

# D.2 안전장치: phys 단독이 더 좋으면 phys만 사용
if best_rh < oof_phys_rh + 0.0001:
    print(">>> Blending gain marginal; using Phys-only for safety")
    final_test = test_phys
    final_w = (1.0, 0.0)
else:
    final_test = best_w * test_phys + (1 - best_w) * test_kal
    final_w = (best_w, 1 - best_w)
print(f"Final weights (phys, kal) = {final_w}")

# D.3 Submission 저장
sub = sample_submission.copy()
# sample_submission의 id 순서와 test_files 순서가 같다고 가정
test_ids = [f.stem for f in test_files]
assert list(sub['id']) == test_ids, "Submission id order mismatch!"
sub[['x','y','z']] = final_test
sub.to_csv('submission.csv', index=False)
print(f"\n✓ submission.csv saved  shape={sub.shape}")
print(sub.head())

# Sanity check
arr = sub[['x','y','z']].values
print(f"\nSanity: nan={np.isnan(arr).any()}, inf={np.isinf(arr).any()}, "
      f"range=[{arr.min():.3f}, {arr.max():.3f}]")
print(f"\nExpected LB: ~{best_rh:.4f} (OOF estimate)")

# 백업: phys-only도 저장
sub_phys = sample_submission.copy()
sub_phys[['x','y','z']] = test_phys
sub_phys.to_csv('submission_phys_only.csv', index=False)
print(f"✓ submission_phys_only.csv saved (fallback)")



Blending
Phys alone OOF: 0.6739
Kal alone OOF:  0.6581
Best blend: w_phys=0.96, w_kal=0.04 → OOF: 0.6744
Final weights (phys, kal) = (np.float64(0.96), np.float64(0.040000000000000036))

✓ submission.csv saved  shape=(10000, 4)
           id         x         y         z
0  TEST_00001  3.989123 -1.052922  0.045457
1  TEST_00002  1.930626  0.616224  0.796840
2  TEST_00003  0.806703 -0.190334  0.146381
3  TEST_00004  3.527112 -1.646297 -0.941020
4  TEST_00005  2.302605  0.016209 -0.039903

Sanity: nan=False, inf=False, range=[-2.602, 6.403]

Expected LB: ~0.6744 (OOF estimate)
✓ submission_phys_only.csv saved (fallback)


# Submission 생성 및 모델 백업

In [ ]:
final_test = best_w * test_phys + (1 - best_w) * test_kal
final_w = (best_w, 1 - best_w)

# D.3 Submission 저장
sub = sample_submission.copy()
# sample_submission의 id 순서와 test_files 순서가 같다고 가정
test_ids = [f.stem for f in test_files]
assert list(sub['id']) == test_ids, "Submission id order mismatch!"
sub[['x','y','z']] = final_test
sub.to_csv('submission(ensemble).csv', index=False)
print(f"\n submission.csv saved  shape={sub.shape}")
print(sub.head())


 submission.csv saved  shape=(10000, 4)
           id         x         y         z
0  TEST_00001  3.989123 -1.052922  0.045457
1  TEST_00002  1.930626  0.616224  0.796840
2  TEST_00003  0.806703 -0.190334  0.146381
3  TEST_00004  3.527112 -1.646297 -0.941020
4  TEST_00005  2.302605  0.016209 -0.039903


In [ ]:
import os
os.makedirs('backups/models/phys', exist_ok=True)
os.makedirs('backups/models/kal', exist_ok=True)
os.makedirs('backups/result', exist_ok=True)

for index, state in enumerate(phys_states):
  torch.save(state, f'backups/models/phys/phys_{index}.pth')

for index, state in enumerate(kal_states):
  torch.save(state, f'backups/models/kal/kal_{index}.pth')

np.save('backups/result/kalman_result.pt', test_kal)
np.save('backups/result/phys_result.pt', test_phys)
np.save('backups/result/ensemble_result.pt', final_test)

print("backup complete")

backup complete


감사합니다.